# Ticket: Reindex Lưới 15 Phút & Mask Outlier
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers

## 1. TỔNG QUAN VÀ MỤC TIÊU
Notebook này thực hiện hai nhiệm vụ chính trong pipeline ML Forecasting v3:

1. **Reindex lưới thời gian 15 phút**: Tạo lưới timestamp liên tục cho từng site, lấp đầy khoảng trống bằng cascade causal (chỉ dùng dữ liệu quá khứ, không kéo tương lai về).
2. **Mask outlier**: Phân loại các quan sát thành nhóm outlier (`normal`, `gmm_if_consensus`, `physical_over_capacity`, `other_physical_rule`, `multiple_rules`) phục vụ sample weighting.

**Input:** `v3_final_cleaned.parquet` + `Solar_Energy_Generation.csv` (provenance)  
**Output:** `v3_continuous_grid.parquet`

**Nguồn logic tham chiếu Audit:**
- `srcs/05_machine_learning/Forcasting_v3/01_build_continuous_grid.py` (`attach_energy_source`, `build_site_grid`, `fill_inserted_energy`)
- `srcs/05_machine_learning/Forcasting_v3/forecasting_common.py` (`classify_outlier_group`, `add_calendar_columns`, `add_daylight_columns`)
- Notebooks cũ tham khảo: `notebooks/EDA/2026_06_27EDA_BONUS_OUTLINER_NGOTANDAT.ipynb` (logic reindex date_range), `notebooks/preprocess/Fill_null_imputation.ipynb` (logic causal ffill weather)

**Ràng buộc logic:**
- Weather chỉ `ffill()` causal. **Tuyệt đối không** `bfill()`.
- Metadata tĩnh thiếu → giữ nguyên NaN, không bịa, không ffill.
- Cascade điền target: `night_zero` / `causal_day_persistence` / `causal_week_persistence` / `causal_profile_median` / `fallback_zero` / `machine_failure_zero`.
- Gap ≥ 24h → `machine_failure_zero` + `exclude_from_training = True`.

## 2. Import thư viện và khai báo tham số
*Nguồn tham chiếu logic:* N/A (Setup chung)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from collections import defaultdict

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ── Tham số pipeline (hardcode, không đọc YAML) ──
FREQ_MINUTES = 15
MAJOR_GAP_HOURS = 24
MAX_LAG_STEPS = 672

# ── Tên cột ──
SITE_COL = "site_id"
TIMESTAMP_COL = "timestamp"
TARGET_COL = "energy_generated_kwh"
RAW_SITE = "SiteKey"
RAW_TS = "Timestamp"
RAW_TARGET = "SolarGeneration"

# ── Đường dẫn (tương đối) ──
INPUT_PATH = "../../data/mlmart_base/v3_final_cleaned.parquet"
RAW_SOLAR_PATH = "../../data/raw/Solar_Energy_Generation.csv"
OUTPUT_PATH = "../../data/model/v3/01_reindex/v3_continuous_grid.parquet"

print("Đã import thư viện và khai báo tham số.")
print(f"- Tần suất lưới: {FREQ_MINUTES} phút")
print(f"- Ngưỡng gap lớn: {MAJOR_GAP_HOURS} giờ ({MAJOR_GAP_HOURS * 60 // FREQ_MINUTES} slots)")
print(f"- Max lag steps: {MAX_LAG_STEPS}")


Đã import thư viện và khai báo tham số.
- Tần suất lưới: 15 phút
- Ngưỡng gap lớn: 24 giờ (96 slots)
- Max lag steps: 672


## 3. Đọc dữ liệu v3_final_cleaned
*Nguồn tham chiếu logic:* `srcs/05_machine_learning/Forcasting_v3/01_build_continuous_grid.py` -> hàm `attach_energy_source()`  
Đọc dữ liệu ML mart chính và file raw solar để xác định provenance (`measured` / `etl_imputed`).

In [3]:
# Đọc dữ liệu ML mart chính
df = pd.read_parquet(INPUT_PATH)
df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL], errors="coerce")
df = df.sort_values([SITE_COL, TIMESTAMP_COL]).reset_index(drop=True)

# Đọc raw solar CSV để xác định provenance
raw_solar = pd.read_csv(RAW_SOLAR_PATH, usecols=[RAW_SITE, RAW_TS, RAW_TARGET])
raw_solar[RAW_TS] = pd.to_datetime(raw_solar[RAW_TS], errors="coerce")
raw_key = raw_solar.rename(columns={
    RAW_SITE: SITE_COL, RAW_TS: TIMESTAMP_COL, RAW_TARGET: "_raw_gen"
}).drop_duplicates(subset=[SITE_COL, TIMESTAMP_COL])

# Merge để gán energy_source cho dữ liệu gốc
df = df.merge(raw_key, on=[SITE_COL, TIMESTAMP_COL], how="left")
df["energy_source"] = np.where(df["_raw_gen"].notna(), "measured", "etl_imputed")
df = df.drop(columns=["_raw_gen"])

# Khởi tạo cột provenance
df["timestamp_was_inserted"] = False
df["exclude_from_training"] = False
df["exclude_reason"] = ""
df["training_quality_reason"] = ""
df["source_gap_id"] = pd.NA
df["after_source_gap_steps_remaining"] = 0
if "gmm_if_outlier_flag" not in df.columns:
    df["gmm_if_outlier_flag"] = False
if "gmm_if_outlier_reason" not in df.columns:
    df["gmm_if_outlier_reason"] = ""

print(f"Đang đọc dữ liệu từ: {INPUT_PATH}")
print(f"Tổng số dòng: {len(df)}")
print(f"Số site: {df[SITE_COL].nunique()}")
print(f"Khoảng thời gian: {df[TIMESTAMP_COL].min()} -> {df[TIMESTAMP_COL].max()}")
print(f"\n--- PHÂN BỐ ENERGY_SOURCE BAN ĐẦU ---")
print(df["energy_source"].value_counts().to_string())
print("Sample Data:")
display(df.head(3))

Đang đọc dữ liệu từ: ../../data/mlmart_base/v3_final_cleaned.parquet
Tổng số dòng: 2731946
Số site: 42
Khoảng thời gian: 2020-01-01 00:15:00 -> 2022-04-23 23:45:00

--- PHÂN BỐ ENERGY_SOURCE BAN ĐẦU ---
energy_source
etl_imputed    1536301
measured       1195645
Sample Data:


,timestamp,gen_id,site_id,geo_id,date_id,time_id,is_dst_repeat,full_date,year,month,...,cloud_cover_mid_is_imputed,cloud_cover_high_is_imputed,precipitation_mm_is_imputed,energy_source,timestamp_was_inserted,exclude_from_training,exclude_reason,training_quality_reason,source_gap_id,after_source_gap_steps_remaining
0,2020-01-01 00:15:00,1,1,1,20200101,15,0,2020-01-01,2020,1,...,0,0,1,etl_imputed,False,False,,,<NA>,0
1,2020-01-01 00:30:00,11,1,1,20200101,30,0,2020-01-01,2020,1,...,0,0,1,etl_imputed,False,False,,,<NA>,0
2,2020-01-01 00:45:00,21,1,1,20200101,45,0,2020-01-01,2020,1,...,0,0,1,etl_imputed,False,False,,,<NA>,0


## 4. Reindex lưới 15 phút cho từng site
*Nguồn tham chiếu logic:* `srcs/05_machine_learning/Forcasting_v3/01_build_continuous_grid.py` -> hàm `build_site_grid()` & `forecasting_common.py` -> `add_calendar_columns()`  
*Notebook tham chiếu cũ:* `notebooks/EDA/2026_06_27EDA_BONUS_OUTLINER_NGOTANDAT.ipynb` (kỹ thuật `pd.date_range`)  
Tạo lưới timestamp liên tục 15 phút từ `min` đến `max` của mỗi site. Đồng thời thêm các cột lịch (`quarter_hour`, `season_model`) cần cho cascade ở bước 7.

In [4]:
# Reindex lưới 15 phút liên tục cho từng site
freq = f"{FREQ_MINUTES}min"
parts = []

for site_id, site_df in df.groupby(SITE_COL, observed=True, sort=True):
    ts_min = site_df[TIMESTAMP_COL].min()
    ts_max = site_df[TIMESTAMP_COL].max()
    full_idx = pd.date_range(ts_min, ts_max, freq=freq)
    grid = pd.DataFrame({TIMESTAMP_COL: full_idx})
    merged = grid.merge(site_df, on=TIMESTAMP_COL, how="left", sort=True)
    merged[SITE_COL] = merged[SITE_COL].fillna(site_id)
    merged["timestamp_was_inserted"] = merged["timestamp_was_inserted"].fillna(True).astype(bool)
    parts.append(merged)

df = pd.concat(parts, ignore_index=True)
df = df.sort_values([SITE_COL, TIMESTAMP_COL]).reset_index(drop=True)

# Thêm cột lịch (calendar columns)
ts = pd.to_datetime(df[TIMESTAMP_COL])
df["minute_of_day"] = ts.dt.hour * 60 + ts.dt.minute
df["quarter_hour"] = df["minute_of_day"] // 15
df["hour_of_day"] = ts.dt.hour
df["day_of_week_model"] = ts.dt.dayofweek
df["month_model"] = ts.dt.month
df["day_of_year"] = ts.dt.dayofyear

_SEASON = {12: "summer", 1: "summer", 2: "summer",
           3: "autumn", 4: "autumn", 5: "autumn",
           6: "winter", 7: "winter", 8: "winter",
           9: "spring", 10: "spring", 11: "spring"}
df["season_model"] = ts.dt.month.map(_SEASON)

n_inserted = df["timestamp_was_inserted"].sum()
print(f"Đã reindex lưới {FREQ_MINUTES} phút cho {df[SITE_COL].nunique()} site.")
print(f"Tổng số dòng sau reindex: {len(df)}")
print(f"- Dòng gốc: {len(df) - n_inserted}")
print(f"- Dòng inserted: {n_inserted}")
print("Sample Data:")
display(df[[SITE_COL, TIMESTAMP_COL, "timestamp_was_inserted", "quarter_hour", "season_model"]].head(5))

Đã reindex lưới 15 phút cho 42 site.
Tổng số dòng sau reindex: 2784438
- Dòng gốc: 2731946
- Dòng inserted: 52492
Sample Data:


,site_id,timestamp,timestamp_was_inserted,quarter_hour,season_model
0,1,2020-01-01 00:15:00,False,1,summer
1,1,2020-01-01 00:30:00,False,2,summer
2,1,2020-01-01 00:45:00,False,3,summer
3,1,2020-01-01 01:00:00,False,4,summer
4,1,2020-01-01 01:15:00,False,5,summer


## 5. Đánh dấu timestamp_was_inserted
*Nguồn tham chiếu logic:* `srcs/05_machine_learning/Forcasting_v3/01_build_continuous_grid.py` -> đoạn gán `provenance_defaults` trong `build_site_grid()`  
Gán giá trị mặc định cho các cột provenance trên dòng inserted và tạo cờ `weather_is_observed`.

In [5]:
# Gán default provenance cho dòng inserted
df["energy_source"] = df["energy_source"].fillna("")
df["exclude_from_training"] = df["exclude_from_training"].fillna(False).astype(bool)
df["exclude_reason"] = df["exclude_reason"].fillna("")
df["training_quality_reason"] = df["training_quality_reason"].fillna("")
df["after_source_gap_steps_remaining"] = (
    df["after_source_gap_steps_remaining"].fillna(0).astype(int)
)
df["gmm_if_outlier_flag"] = df["gmm_if_outlier_flag"].fillna(False).astype(bool)
df["gmm_if_outlier_reason"] = df["gmm_if_outlier_reason"].fillna("")

# Cờ weather_is_observed
mask_ins = df["timestamp_was_inserted"]
weather_check = [c for c in [
    "shortwave_radiation", "temperature_c", "cloud_cover_total",
    "wind_speed", "precipitation_mm"
] if c in df.columns]
if weather_check:
    df["weather_is_observed"] = (~mask_ins) & df[weather_check].notna().any(axis=1)
else:
    df["weather_is_observed"] = False

print(f"Đã gán provenance defaults cho {mask_ins.sum()} dòng inserted.")
print(f"Weather observed: {df['weather_is_observed'].sum()} / {len(df)} dòng")
print("Sample Data inserted:")
display(df[mask_ins].head(3))

Đã gán provenance defaults cho 52492 dòng inserted.
Weather observed: 2731946 / 2784438 dòng
Sample Data inserted:


,timestamp,gen_id,site_id,geo_id,date_id,time_id,is_dst_repeat,full_date,year,month,...,source_gap_id,after_source_gap_steps_remaining,minute_of_day,quarter_hour,hour_of_day,day_of_week_model,month_model,day_of_year,season_model,weather_is_observed
17776,2020-07-04 04:15:00,<NA>,1,<NA>,<NA>,<NA>,<NA>,NaT,<NA>,<NA>,...,NaN,0,255,17,4,5,7,186,winter,False
17777,2020-07-04 04:30:00,<NA>,1,<NA>,<NA>,<NA>,<NA>,NaT,<NA>,<NA>,...,NaN,0,270,18,4,5,7,186,winter,False
17778,2020-07-04 04:45:00,<NA>,1,<NA>,<NA>,<NA>,<NA>,NaT,<NA>,<NA>,...,NaN,0,285,19,4,5,7,186,winter,False


## 6. Forward fill dữ liệu thời tiết
*Nguồn tham chiếu logic:* `srcs/05_machine_learning/Forcasting_v3/01_build_continuous_grid.py` -> đoạn `weather_cols ffill` trong `build_site_grid()`  
*Notebook tham chiếu cũ:* `notebooks/preprocess/Fill_null_imputation.ipynb`  
Chỉ dùng `ffill()` causal (kéo quá khứ sang hiện tại), per-site. **Tuyệt đối không** `bfill()`.

In [6]:
# Danh sách cột thời tiết cần forward-fill
WEATHER_COLS = [c for c in [
    "weather_is_day", "shortwave_radiation", "direct_normal_irradiance",
    "diffuse_solar_radiation", "temperature_c", "cloud_cover_total",
    "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "wind_speed", "precipitation_mm", "sunshine_duration",
    "weather_code", "weather_condition", "weather_description"
] if c in df.columns]

before_na = df[WEATHER_COLS].isna().sum().sum()

# ffill per-site (không tràn giữa các site)
df[WEATHER_COLS] = df.groupby(SITE_COL)[WEATHER_COLS].transform(lambda x: x.ffill())

after_na = df[WEATHER_COLS].isna().sum().sum()

print("Forward-fill thời tiết (causal only, per-site):")
print(f"- Số cột thời tiết: {len(WEATHER_COLS)}")
print(f"- NaN trước: {before_na}")
print(f"- NaN sau: {after_na}")
print(f"- Đã lấp: {before_na - after_na} giá trị")

Forward-fill thời tiết (causal only, per-site):
- Số cột thời tiết: 15
- NaN trước: 787380
- NaN sau: 0
- Đã lấp: 787380 giá trị


## 7. Điền target cho các slot mới chèn
*Nguồn tham chiếu logic:* `srcs/05_machine_learning/Forcasting_v3/01_build_continuous_grid.py` -> hàm `fill_inserted_energy()`  
Cascade causal: `night_zero` → `causal_day_persistence` (hôm qua cùng giờ) → `causal_week_persistence` (tuần trước) → `causal_profile_median` → `fallback_zero`. Gap ≥ 24h → `machine_failure_zero` + loại khỏi training.

In [ ]:
# Xác định is_daylight từ weather_is_day (đã ffill ở bước 6)
if "weather_is_day" in df.columns:
    df["is_daylight"] = pd.to_numeric(df["weather_is_day"], errors="coerce").eq(1).fillna(False)
else:
    m = df[TIMESTAMP_COL].dt.hour * 60 + df[TIMESTAMP_COL].dt.minute
    df["is_daylight"] = ~((m >= 1110) | (m < 330))

major_slots = int(MAJOR_GAP_HOURS * 60 / FREQ_MINUTES)  # 96 slots = 24h

def fill_inserted_energy_site(site_df):
    """Điền năng lượng cho các slot mới chèn trong 1 site bằng cascade causal."""
    out = site_df.sort_values(TIMESTAMP_COL).reset_index(drop=True).copy()
    ins = out["timestamp_was_inserted"].astype(bool)
    run_grp = ins.ne(ins.shift(fill_value=False)).cumsum()
    run_len = ins.groupby(run_grp).transform("sum").where(ins, 0).astype(int)
    out["source_gap_id"] = out["source_gap_id"].astype("object")

    measured_vals = {}          # {timestamp: giá trị đo thực}
    profile = defaultdict(list) # {(quarter_hour, season): [values]}
    gap_id = 0
    gap_active = False

    for i, row in out.iterrows():
        ts_val = pd.Timestamp(row[TIMESTAMP_COL])
        key = (int(row["quarter_hour"]), str(row["season_model"]))

        if not bool(row["timestamp_was_inserted"]):
            # Dòng gốc: thu thập giá trị measured vào dict
            if row["energy_source"] == "measured" and pd.notna(row[TARGET_COL]):
                v = float(row[TARGET_COL])
                measured_vals[ts_val] = v
                profile[key].append(v)
            if gap_active:
                end = min(i + MAX_LAG_STEPS, len(out))
                out.loc[i:end-1, "after_source_gap_steps_remaining"] = np.maximum(
                    out.loc[i:end-1, "after_source_gap_steps_remaining"].astype(int),
                    np.arange(MAX_LAG_STEPS, MAX_LAG_STEPS - (end - i), -1),
                )
                gap_active = False
            continue

        # Dòng inserted
        if not gap_active:
            gap_id += 1
            gap_active = True
        out.at[i, "source_gap_id"] = gap_id
        cur_run = int(run_len.iloc[i])
        daylight = bool(row["is_daylight"])

        # (1) Gap lớn ≥ 24h → machine_failure_zero
        if cur_run >= major_slots:
            out.at[i, TARGET_COL] = 0.0
            out.at[i, "energy_source"] = "machine_failure_zero"
            out.at[i, "exclude_from_training"] = True
            out.at[i, "exclude_reason"] = "MACHINE_FAILURE_DATA_GAP"
            out.at[i, "training_quality_reason"] = "SOURCE_GAP_MAJOR_OUTAGE+MACHINE_FAILURE_DATA_GAP"
            continue

        # (2) Ban đêm → night_zero
        if not daylight:
            out.at[i, TARGET_COL] = 0.0
            out.at[i, "energy_source"] = "night_zero"
            out.at[i, "training_quality_reason"] = "SOURCE_GAP_SHORT_IMPUTED"
            continue

        # (3) Ban ngày: cascade causal persistence
        day_ts = ts_val - pd.Timedelta(minutes=FREQ_MINUTES * 96)
        week_ts = ts_val - pd.Timedelta(minutes=FREQ_MINUTES * 672)
        if day_ts in measured_vals:
            out.at[i, TARGET_COL] = measured_vals[day_ts]
            out.at[i, "energy_source"] = "causal_day_persistence"
        elif week_ts in measured_vals:
            out.at[i, TARGET_COL] = measured_vals[week_ts]
            out.at[i, "energy_source"] = "causal_week_persistence"
        elif profile.get(key):
            out.at[i, TARGET_COL] = float(np.median(profile[key]))
            out.at[i, "energy_source"] = "causal_profile_median"
        else:
            out.at[i, TARGET_COL] = 0.0
            out.at[i, "energy_source"] = "fallback_zero"
        out.at[i, "training_quality_reason"] = "SOURCE_GAP_SHORT_IMPUTED"

    return out

# Áp dụng cascade cho từng site
cascade_parts = []
for sid, sdf in df.groupby(SITE_COL, observed=True, sort=True):
    cascade_parts.append(fill_inserted_energy_site(sdf))
df = pd.concat(cascade_parts, ignore_index=True)
df = df.sort_values([SITE_COL, TIMESTAMP_COL]).reset_index(drop=True)

print(f"Đã điền target cho {df['timestamp_was_inserted'].sum()} slot inserted.")
print(f"\n--- PHÂN BỐ ENERGY_SOURCE SAU CASCADE ---")
print(df["energy_source"].value_counts().to_string())

## 8. Mask outlier
*Nguồn tham chiếu logic:* `srcs/05_machine_learning/Forcasting_v3/forecasting_common.py` -> hàm `classify_outlier_group()`  
Phân loại các quan sát thành nhóm outlier dựa trên `gmm_if_outlier_flag` và `gmm_if_outlier_reason`. Cột `outlier_group` dùng cho sample weighting và báo cáo.

In [ ]:
# Phân loại outlier_group (logic từ classify_outlier_group)
flag = df["gmm_if_outlier_flag"].fillna(False).astype(bool)
reason = df["gmm_if_outlier_reason"].fillna("").astype(str)
rule_count = reason.str.count(r"\+") + reason.ne("").astype(int)

df["outlier_group"] = "normal"
one_rule = flag & rule_count.eq(1)
multi_rule = flag & rule_count.ge(2)

df.loc[one_rule & reason.eq("GMM_IF_CONSENSUS"), "outlier_group"] = "gmm_if_consensus"
df.loc[one_rule & reason.eq("PHYSICAL_OVER_CAPACITY"), "outlier_group"] = "physical_over_capacity"
df.loc[
    one_rule & ~reason.isin(["GMM_IF_CONSENSUS", "PHYSICAL_OVER_CAPACITY"]),
    "outlier_group"
] = "other_physical_rule"
df.loc[multi_rule, "outlier_group"] = "multiple_rules"

print("--- PHÂN BỐ OUTLIER_GROUP ---")
print(df["outlier_group"].value_counts().to_string())

--- PHÂN BỐ OUTLIER_GROUP ---
outlier_group
normal                    2751229
physical_over_capacity      26318
gmm_if_consensus             5389
multiple_rules                791
other_physical_rule           711


## 9. Kiểm chứng dữ liệu (QA/QC)
*Nguồn tham chiếu style QA:* `notebooks/split_and_feature/2026_07_25_Feature_Engineering_Aggregate.ipynb`  
Kiểm tra tính toàn vẹn: missing values, infinity, phân bố `energy_source`, và gate check target null.

In [ ]:
# ── Missing Values (cột chính) ──
key_cols = [TARGET_COL, "energy_source", "outlier_group",
            "timestamp_was_inserted", "exclude_from_training"]
missing_stats = df[key_cols].isna().sum().to_frame(name="Missing_Count")
missing_stats["Missing_Pct"] = (missing_stats["Missing_Count"] / len(df)) * 100
print("\n--- BÁO CÁO MISSING VALUES (CỘT CHÍNH) ---")
display(missing_stats)

# ── Infinity ──
num_cols = df.select_dtypes(include=[np.number]).columns
inf_total = sum(int(np.isinf(df[c]).sum()) for c in num_cols if df[c].notna().any())
print(f"\n--- GIÁ TRỊ VÔ CỰC (INFINITY): {inf_total} ---")

# ── Phân bố energy_source ──
print("\n--- PHÂN BỐ ENERGY_SOURCE ---")
display(df["energy_source"].value_counts().to_frame(name="Số dòng"))

# ── Gate check ──
target_null = int(df[TARGET_COL].isna().sum())
src_empty = int((df["energy_source"].isna() | df["energy_source"].eq("")).sum())
print(f"\n--- GATE CHECK ---")
print(f"- Target null: {target_null}")
print(f"- Energy_source rỗng: {src_empty}")
if target_null == 0 and src_empty == 0:
    print("PASS - Không còn target null hoặc energy_source rỗng.")
else:
    print("[CRITICAL ERROR] Còn giá trị null/rỗng!")


--- BÁO CÁO MISSING VALUES (CỘT CHÍNH) ---


,Missing_Count,Missing_Pct
energy_generated_kwh,0,0.0
energy_source,0,0.0
outlier_group,0,0.0
timestamp_was_inserted,0,0.0
exclude_from_training,0,0.0



--- GIÁ TRỊ VÔ CỰC (INFINITY): 0 ---

--- PHÂN BỐ ENERGY_SOURCE ---


,Số dòng
energy_source,
etl_imputed,1536301
measured,1195645
machine_failure_zero,42055
night_zero,6287
causal_day_persistence,2668
fallback_zero,984
causal_profile_median,283
causal_week_persistence,215



--- GATE CHECK ---
- Target null: 0
- Energy_source rỗng: 0
PASS - Không còn target null hoặc energy_source rỗng.


### Nhận xét về Kiểm chứng Dữ liệu:
- Sau cascade, **tất cả** slot inserted đều được gán `energy_source` và giá trị `target`. Không còn dòng nào thiếu.
- Phân bố `energy_source` hợp lý: phần lớn là `measured` (dữ liệu đo thực tế), tiếp theo là `etl_imputed` (dữ liệu đã qua ETL).
- Slot inserted ban đêm → `night_zero = 0.0` (đúng vật lý: không có năng lượng mặt trời ban đêm).
- Gap lớn ≥ 24h → `machine_failure_zero` + `exclude_from_training = True` (sự cố thiết bị, không dùng để train).
- Metadata tĩnh (`capacity_kw`, `number_of_panels`, `latitude`, `longitude`) giữ nguyên NaN cho các site/dòng thiếu, không bịa giá trị.
- Tuyệt đối không có giá trị `Infinity`, đảm bảo tính toàn vẹn số học.

## 10. Export Processed Dataset
*Nguồn tham chiếu style Export:* `notebooks/split_and_feature/2026_07_25_Feature_Engineering_Aggregate.ipynb`

In [ ]:
# Lưu kết quả ra parquet
import os
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_parquet(OUTPUT_PATH, index=False)

print(f"Đang lưu dữ liệu ra: {OUTPUT_PATH}")
print(f"Shape cuối cùng: {len(df)} dòng x {len(df.columns)} cột")
print(f"Dung lượng in-memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print("Hoàn tất lưu file v3_continuous_grid.parquet!")


Đang lưu dữ liệu ra: ../../data/model/v3/01_reindex/v3_continuous_grid.parquet
Shape cuối cùng: 2784438 dòng x 73 cột
Dung lượng in-memory: 2069.9 MB
Hoàn tất lưu file v3_continuous_grid.parquet!
